In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# Function to convert Excel serial date to datetime (if needed)
def excel_to_datetime(serial):
    try:
        serial = int(serial)
        return datetime(1899, 12, 30) + timedelta(days=serial)
    except:
        return pd.NaT

In [ ]:
# Step 2: Load and Inspect KSE Data (unchanged)
file_path = 'data/kse-30-basic.xlsx'
df_org = pd.read_excel(file_path)
df = df_org.copy()
df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')
if not pd.api.types.is_datetime64_any_dtype(df['date']):
    df['date'] = df['date'].apply(excel_to_datetime)
df = df.dropna(subset=['date', 'company', 'price', 'idx_wt_%', 'volume'])
df = df.sort_values(['date', 'company']).reset_index(drop=True)
df_cleaned = df.copy()

In [ ]:
df_cleaned.head()

In [ ]:
# Compute daily aggregates
df_cleaned['weighted_price'] = df_cleaned['price'] * df_cleaned['idx_wt_%']
daily_agg = df_cleaned.groupby('date').agg(
    total_volume=('volume', 'sum'),
    weighted_return=('weighted_price', 'sum')
).reset_index()

In [ ]:
df_cleaned.head()

In [ ]:
daily_agg.head()

In [ ]:
# Resample to monthly
daily_agg['date'] = pd.to_datetime(daily_agg['date'])
daily_agg.set_index('date', inplace=True)
monthly_df = daily_agg.resample('ME').agg({
    'total_volume': 'sum',
    'weighted_return': 'mean'
}).reset_index()

monthly_df['log_return'] = np.log(monthly_df['weighted_return'] / monthly_df['weighted_return'].shift(1))

In [ ]:
monthly_df.head()

In [ ]:
# Load and process funds_data for flows
fund_sheets = ['AKD', 'NBP', 'NTI']
fund_flows = []
for sheet in fund_sheets:
    fund_df = pd.read_excel('data/funds_data.xlsx', sheet_name=sheet)
    fund_df.columns = fund_df.columns.str.lower().str.strip().str.replace(' ', '_')
    if not pd.api.types.is_datetime64_any_dtype(fund_df['date']):
        fund_df['date'] = fund_df['date'].apply(excel_to_datetime)
    fund_df = fund_df.dropna(subset=['date', 'nav', 'aum'])
    fund_df = fund_df.sort_values('date').reset_index(drop=True)
    fund_df['nav_ratio'] = fund_df['nav'] / fund_df['nav'].shift(1)
    fund_df['flow'] = fund_df['aum'] - (fund_df['aum'].shift(1) * fund_df['nav_ratio'])
    fund_df['flow'] = fund_df['flow'].fillna(0)
    fund_flows.append(fund_df[['date', 'flow']])

all_fund_flows = pd.concat(fund_flows)
all_fund_flows = all_fund_flows.groupby('date')['flow'].sum().reset_index()
all_fund_flows['date'] = pd.to_datetime(all_fund_flows['date'])
all_fund_flows.set_index('date', inplace=True)
total_monthly_flow = all_fund_flows.resample('ME')['flow'].sum().reset_index(name='total_fund_flow')

In [ ]:
all_fund_flows.head()

In [ ]:
total_monthly_flow.head()

In [ ]:
# Load and process macro_data
oil_df = pd.read_excel('data/macro_data.xlsx', sheet_name='OIL')
oil_df.columns = oil_df.columns.str.lower().str.strip().str.replace(' ', '_')
if not pd.api.types.is_datetime64_any_dtype(oil_df['date']):
    oil_df['date'] = oil_df['date'].apply(excel_to_datetime)
oil_df = oil_df.dropna(subset=['date'])
oil_df = oil_df.groupby('date')['price'].mean().reset_index()
oil_df['date'] = pd.to_datetime(oil_df['date'])
oil_df.set_index('date', inplace=True)
monthly_oil = oil_df.resample('ME')['price'].mean().reset_index(name='oil_price')

In [ ]:
oil_df.head()

In [ ]:
monthly_oil.head()

In [ ]:
ir_df = pd.read_excel('data/macro_data.xlsx', sheet_name='IR')
ir_df.columns = ir_df.columns.str.lower().str.strip().str.replace(' ', '_')
if not pd.api.types.is_datetime64_any_dtype(ir_df['date']):
    ir_df['date'] = ir_df['date'].apply(excel_to_datetime)
ir_df = ir_df.dropna(subset=['date'])
ir_df = ir_df.groupby('date')['rate'].last().reset_index()
ir_df['date'] = pd.to_datetime(ir_df['date'])
ir_df.set_index('date', inplace=True)
ir_df.sort_index(inplace=True)
daily_index = pd.date_range(start=ir_df.index.min(), end=ir_df.index.max(), freq='D')
ir_df = ir_df.reindex(daily_index).ffill()
monthly_ir = ir_df.resample('ME')['rate'].last().reset_index()
if 'index' in monthly_ir.columns:
    monthly_ir.rename(columns={'index': 'date'}, inplace=True)
monthly_ir.rename(columns={'rate': 'interest_rate'}, inplace=True)

In [ ]:
ir_df.head()

In [ ]:
monthly_ir.head()

In [ ]:
usd_df = pd.read_excel('data/macro_data.xlsx', sheet_name='USD')
usd_df.columns = usd_df.columns.str.lower().str.strip().str.replace(' ', '_')
if not pd.api.types.is_datetime64_any_dtype(usd_df['date']):
    usd_df['date'] = usd_df['date'].apply(excel_to_datetime)
usd_df = usd_df.dropna(subset=['date'])
usd_df = usd_df.groupby('date')['usd'].mean().reset_index()
usd_df['date'] = pd.to_datetime(usd_df['date'])
usd_df.set_index('date', inplace=True)
monthly_usd = usd_df.resample('ME')['usd'].mean().reset_index(name='usd_exchange')

In [ ]:
usd_df.head()

In [ ]:
monthly_usd.head()

In [ ]:
# Merge to enhanced monthly df
enhanced_monthly = monthly_df.merge(total_monthly_flow, on='date', how='left')
enhanced_monthly = enhanced_monthly.merge(monthly_oil, on='date', how='left')
enhanced_monthly = enhanced_monthly.merge(monthly_ir, on='date', how='left')
enhanced_monthly = enhanced_monthly.merge(monthly_usd, on='date', how='left')

# Fund flow: months before fund data starts have no flow — treat as 0
enhanced_monthly['total_fund_flow'] = enhanced_monthly['total_fund_flow'].fillna(0)

# Macro columns (oil, interest rate, USD) start from 2021.
# bfill fills pre-2021 rows with the earliest real value we have;
# ffill then closes any remaining gaps within the series.
enhanced_monthly[['oil_price', 'interest_rate', 'usd_exchange']] = (
    enhanced_monthly[['oil_price', 'interest_rate', 'usd_exchange']]
    .bfill()
    .ffill()
)

# log_return for the first row is NaN (no prior month) — fill with 0
enhanced_monthly['log_return'] = enhanced_monthly['log_return'].fillna(0)

In [ ]:
enhanced_monthly.head(10)

In [ ]:
# Winsorize total_fund_flow at ±3 standard deviations
# The April 2022 outlier (-62.7) is a one-off political event, not a learnable pattern
mean_flow = enhanced_monthly['total_fund_flow'].mean()
std_flow = enhanced_monthly['total_fund_flow'].std()
lower_bound = mean_flow - 3 * std_flow
upper_bound = mean_flow + 3 * std_flow
enhanced_monthly['total_fund_flow'] = enhanced_monthly['total_fund_flow'].clip(lower=lower_bound, upper=upper_bound)
print(f"Winsorize bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
print(f"Flow range after winsorize: [{enhanced_monthly['total_fund_flow'].min():.2f}, {enhanced_monthly['total_fund_flow'].max():.2f}]")

In [ ]:
# Lagged level features
enhanced_monthly['lag_volume'] = enhanced_monthly['total_volume'].shift(1)
enhanced_monthly['lag_return'] = enhanced_monthly['log_return'].shift(1)
enhanced_monthly['lag_oil'] = enhanced_monthly['oil_price'].shift(1)
enhanced_monthly['lag_ir'] = enhanced_monthly['interest_rate'].shift(1)
enhanced_monthly['lag_usd'] = enhanced_monthly['usd_exchange'].shift(1)

# Add macro change features to improve signal capture
enhanced_monthly['ir_change'] = enhanced_monthly['interest_rate'].diff()
enhanced_monthly['usd_pct_change'] = enhanced_monthly['usd_exchange'].pct_change()
enhanced_monthly['lag_ir_change'] = enhanced_monthly['ir_change'].shift(1)
enhanced_monthly['lag_usd_pct_change'] = enhanced_monthly['usd_pct_change'].shift(1)

enhanced_monthly = enhanced_monthly.dropna()

In [ ]:
enhanced_monthly.head()

In [ ]:
features = ['lag_volume', 'lag_return', 'lag_oil', 'lag_ir', 'lag_usd',
            'lag_ir_change', 'lag_usd_pct_change']
X = enhanced_monthly[features]
y = enhanced_monthly['total_fund_flow']

tscv = TimeSeriesSplit(n_splits=5)

ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=1.0))
])

fold_rmses = []
for fold, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    ridge_pipeline.fit(X_train, y_train)
    preds = ridge_pipeline.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    fold_rmses.append(rmse)
    print(f"Fold {fold} | test rows: {len(y_test)} | RMSE: {rmse:.4f}")

print(f"\nMean RMSE across folds: {np.mean(fold_rmses):.4f}")
print(f"Std  RMSE across folds: {np.std(fold_rmses):.4f}")

# Refit on full data for next-month prediction
ridge_pipeline.fit(X, y)
last_row = enhanced_monthly.iloc[-1][features].values.reshape(1, -1)
next_pred = ridge_pipeline.predict(last_row)
print(f"\nPredicted next fund flow (Ridge): {next_pred[0]:.4f}")

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(12, 18), sharex=False)
fig.suptitle('Fund Flow Predictions - Ridge + TimeSeriesSplit (5 Folds)', fontsize=14, y=1.01)

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    ridge_plot = Pipeline([
        ('scaler', StandardScaler()),
        ('ridge', Ridge(alpha=1.0))
    ])
    ridge_plot.fit(X_train, y_train)
    preds = ridge_plot.predict(X_test)

    ax = axes[fold - 1]
    ax.plot(enhanced_monthly['date'].iloc[test_idx], y_test.values, marker='o', label='Actual', color='steelblue')
    ax.plot(enhanced_monthly['date'].iloc[test_idx], preds, marker='x', linestyle='--', label='Predicted (Ridge)', color='tomato')
    ax.set_title(f'Fold {fold} | RMSE: {fold_rmses[fold-1]:.4f}')
    ax.set_ylabel('Fund Flow')
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Train/test split
split = int(len(enhanced_monthly) * 0.8)
train = enhanced_monthly.iloc[:split]
test = enhanced_monthly.iloc[split:]

features = ['lag_volume', 'lag_return', 'lag_oil', 'lag_ir', 'lag_usd',
            'lag_ir_change', 'lag_usd_pct_change']
X_train, y_train = train[features], train['total_fund_flow']
X_test, y_test = test[features], test['total_fund_flow']

# Ridge model pipeline
ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=1.0))
])
ridge_pipeline.fit(X_train, y_train)
preds_ridge = ridge_pipeline.predict(X_test)

# Evaluate
mse = mean_squared_error(y_test, preds_ridge)
rmse = np.sqrt(mse)
print(f"RMSE on test (Ridge): {rmse:.4f}")

# Predict next
last_row = enhanced_monthly.iloc[-1][features].values.reshape(1, -1)
next_pred = ridge_pipeline.predict(last_row)
print(f"Predicted next fund flow (Ridge): {next_pred[0]:.4f}")

# Plot predictions (optional)
plt.figure(figsize=(10, 5))
plt.plot(test['date'], y_test, label='Actual Flows')
plt.plot(test['date'], preds_ridge, label='Predicted (Ridge)')
plt.title('Fund Flow Prediction (Test Set) - Ridge')
plt.legend()
plt.show()

r2 = r2_score(y_test, preds_ridge)
print(f"R-squared on test: {r2:.4f}")

n = len(y_test)          # number of observations
p = X_test.shape[1]      # number of predictors
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
print(f"Adjusted R-squared on test: {adj_r2:.4f}")

In [ ]:
# Suggestion
threshold = enhanced_monthly['total_fund_flow'].mean()
if next_pred[0] > threshold:
    print("Predicted inflow → Suggest overweight high-momentum/volume stocks")
    recent_date = df_cleaned['date'].max()
    recent = df_cleaned[df_cleaned['date'] == recent_date].sort_values('volume', ascending=False).head(10)
    print(recent[['company', 'volume', 'idx_wt_%']])
else:
    print("Predicted outflow/flat → Suggest defensive or equal-weight")

In [ ]:
# New final cell: inspect Ridge coefficients for interpretability
coef = ridge_pipeline.named_steps['ridge'].coef_
coef_df = pd.DataFrame({
    'feature': features,
    'coefficient': coef,
    'abs_coefficient': np.abs(coef)
}).sort_values('abs_coefficient', ascending=False)

print('Top drivers by absolute Ridge coefficient:')
print(coef_df[['feature', 'coefficient']].to_string(index=False))

In [ ]:
# XGBoost model for fund flow prediction
# If xgboost is missing, install in this kernel: %pip install xgboost
from xgboost import XGBRegressor

xgb_features = ['lag_volume', 'lag_return', 'lag_oil', 'lag_ir', 'lag_usd',
                'lag_ir_change', 'lag_usd_pct_change']
X_xgb = enhanced_monthly[xgb_features]
y_xgb = enhanced_monthly['total_fund_flow']

split_xgb = int(len(enhanced_monthly) * 0.8)
X_train_xgb, X_test_xgb = X_xgb.iloc[:split_xgb], X_xgb.iloc[split_xgb:]
y_train_xgb, y_test_xgb = y_xgb.iloc[:split_xgb], y_xgb.iloc[split_xgb:]

xgb_model = XGBRegressor(
    objective='reg:squarederror',
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)

xgb_model.fit(X_train_xgb, y_train_xgb)
preds_xgb = xgb_model.predict(X_test_xgb)

rmse_xgb = np.sqrt(mean_squared_error(y_test_xgb, preds_xgb))
r2_xgb = r2_score(y_test_xgb, preds_xgb)
print(f'XGBoost RMSE on test: {rmse_xgb:.4f}')
print(f'XGBoost R-squared on test: {r2_xgb:.4f}')

next_pred_xgb = xgb_model.predict(X_xgb.iloc[[-1]])
print(f'Predicted next fund flow (XGBoost): {next_pred_xgb[0]:.4f}')

In [ ]:
# Visualize XGBoost predictions and high-volume stocks
pred_dates = enhanced_monthly['date'].iloc[split_xgb:].reset_index(drop=True)
actual_test = y_test_xgb.reset_index(drop=True)
pred_test = pd.Series(preds_xgb)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: test-set prediction vs actual
axes[0].plot(pred_dates, actual_test, marker='o', label='Actual', color='steelblue')
axes[0].plot(pred_dates, pred_test, marker='x', linestyle='--', label='Predicted (XGBoost)', color='tomato')
axes[0].set_title('XGBoost Test Prediction vs Actual')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Fund Flow')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend()

# Right: latest high-volume stocks
latest_date = df_cleaned['date'].max()
top_volume = (
    df_cleaned[df_cleaned['date'] == latest_date]
    .sort_values('volume', ascending=False)
    .head(10)
)

axes[1].barh(top_volume['company'], top_volume['volume'], color='teal')
axes[1].invert_yaxis()
axes[1].set_title(f'Top 10 High-Volume Stocks ({latest_date.date()})')
axes[1].set_xlabel('Volume')
axes[1].set_ylabel('Company')

plt.tight_layout()
plt.show()

print('Top 10 high-volume stocks (latest date):')
print(top_volume[['company', 'volume', 'idx_wt_%']].to_string(index=False))

In [ ]:
# Comprehensive metrics comparison: Ridge vs XGBoost
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

# Ridge metrics
ridge_rmse = np.sqrt(mean_squared_error(y_test, preds_ridge))
ridge_mae = mean_absolute_error(y_test, preds_ridge)
ridge_r2 = r2_score(y_test, preds_ridge)
ridge_mape = mean_absolute_percentage_error(y_test, preds_ridge) * 100

# XGBoost metrics  
xgb_rmse = np.sqrt(mean_squared_error(y_test_xgb, preds_xgb))
xgb_mae = mean_absolute_error(y_test_xgb, preds_xgb)
xgb_r2 = r2_score(y_test_xgb, preds_xgb)
xgb_mape = mean_absolute_percentage_error(y_test_xgb, preds_xgb) * 100

# Directional accuracy (sign prediction)
ridge_direction_acc = np.mean(np.sign(y_test) == np.sign(preds_ridge)) * 100
xgb_direction_acc = np.mean(np.sign(y_test_xgb.values) == np.sign(preds_xgb)) * 100

# Create metrics comparison table
metrics_df = pd.DataFrame({
    'Metric': ['RMSE', 'MAE', 'R²', 'MAPE (%)', 'Directional Accuracy (%)'],
    'Ridge': [ridge_rmse, ridge_mae, ridge_r2, ridge_mape, ridge_direction_acc],
    'XGBoost': [xgb_rmse, xgb_mae, xgb_r2, xgb_mape, xgb_direction_acc]
})

print('=' * 60)
print('MODEL PERFORMANCE COMPARISON - Test Set')
print('=' * 60)
print(metrics_df.to_string(index=False))
print('=' * 60)
print(f'\nRidge next prediction: {next_pred[0]:.4f}')
print(f'XGBoost next prediction: {next_pred_xgb[0]:.4f}')
print(f'\nNote: Lower RMSE/MAE/MAPE and higher R²/Directional Accuracy indicate better performance')

In [ ]:
# LSTM model for fund flow prediction
# Install if needed: %pip install tensorflow
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler

# Prepare data for LSTM (needs 3D: samples, timesteps, features)
lstm_features = ['lag_volume', 'lag_return', 'lag_oil', 'lag_ir', 'lag_usd',
                 'lag_ir_change', 'lag_usd_pct_change']
X_lstm = enhanced_monthly[lstm_features].values
y_lstm = enhanced_monthly['total_fund_flow'].values

# Scale features
scaler_lstm = StandardScaler()
X_lstm_scaled = scaler_lstm.fit_transform(X_lstm)

# Reshape for LSTM: (samples, timesteps=1, features)
X_lstm_3d = X_lstm_scaled.reshape((X_lstm_scaled.shape[0], 1, X_lstm_scaled.shape[1]))

# Train/test split
split_lstm = int(len(X_lstm_3d) * 0.8)
X_train_lstm, X_test_lstm = X_lstm_3d[:split_lstm], X_lstm_3d[split_lstm:]
y_train_lstm, y_test_lstm = y_lstm[:split_lstm], y_lstm[split_lstm:]

# Build LSTM model
tf.random.set_seed(42)
lstm_model = Sequential([
    LSTM(32, activation='tanh', return_sequences=False, input_shape=(1, len(lstm_features))),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])

lstm_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Train with early stopping
early_stop = EarlyStopping(monitor='loss', patience=20, restore_best_weights=True)
history = lstm_model.fit(
    X_train_lstm, y_train_lstm,
    epochs=100,
    batch_size=8,
    verbose=0,
    callbacks=[early_stop]
)

# Predictions
preds_lstm = lstm_model.predict(X_test_lstm, verbose=0).flatten()
lstm_rmse = np.sqrt(mean_squared_error(y_test_lstm, preds_lstm))
lstm_r2 = r2_score(y_test_lstm, preds_lstm)
lstm_direction_acc = np.mean(np.sign(y_test_lstm) == np.sign(preds_lstm)) * 100

print('=' * 60)
print('LSTM MODEL PERFORMANCE')
print('=' * 60)
print(f'RMSE: {lstm_rmse:.4f}')
print(f'R²: {lstm_r2:.4f}')
print(f'Directional Accuracy: {lstm_direction_acc:.2f}%')

# Next prediction
next_lstm = lstm_model.predict(X_lstm_3d[-1:], verbose=0)[0][0]
print(f'\nLSTM next prediction: {next_lstm:.4f}')

print('\n' + '=' * 60)
print('SUGGESTIONS TO IMPROVE PREDICTIONS')
print('=' * 60)
print("""
1. DATA AUGMENTATION
   - Add more macro indicators: inflation, GDP growth, political stability index
   - Include sentiment data: news sentiment, social media trends
   - Add sector-level data: sector rotation, relative strength
   - Get more historical data (currently ~50 months → aim for 5+ years)

2. FEATURE ENGINEERING
   - Create momentum indicators: 3/6/12-month moving averages
   - Add rolling volatility measures (fund flow and market)
   - Technical indicators: RSI, MACD for index prices
   - Seasonality features: month, quarter dummies
   - Lagged target: past 3/6 months of fund flows

3. MODEL IMPROVEMENTS
   - Ensemble: Combine Ridge, XGBoost, LSTM predictions (weighted average)
   - Sequence modeling: Increase LSTM timesteps (use sequences of 3-6 months)
   - Attention mechanisms: Add attention layers to LSTM
   - Hyperparameter tuning: Grid/random search for optimal params

4. DATA QUALITY
   - Remove/interpolate outliers more carefully (beyond winsorization)
   - Cross-validate imputation methods for missing macro data
   - Use forward-looking data only (avoid look-ahead bias)

5. VALIDATION STRATEGY
   - Walk-forward validation: retrain monthly with expanding window
   - Out-of-sample testing: hold out last 12 months completely
   - Monte Carlo simulation: test robustness with bootstrapping

6. DOMAIN KNOWLEDGE
   - Incorporate known catalysts: policy changes, geopolitical events
   - Add fund-specific features: expense ratios, manager tenure
   - Market regime indicators: bull/bear/sideways classification

7. FOCUS: Use predictions for DIRECTION (inflow/outflow) rather than
   exact magnitude — classification models (Logistic/Random Forest Classifier)
   may perform better for binary directional signals.
""")